NOTEBOOK INGESTION_SILVER

Responsável por mover os dados brutos para a camada silver, fazer a limpeza e tratamento dos dados e adicionar colunas de auditoria. 

# CONFIGURAÇÕES GERAIS

## Rodar notebooks de configuração

In [0]:
%run ../../config/feat_squad2_config_adls

In [0]:
%run ../../utils/feat_squad2_utils

## Configurar as variáveis

In [0]:
entity_name = dbutils.widgets.get("entity_name")
folder_name_source = f"bronze/ecommerce_{entity_name}"
path_source = f"abfs://{container_name_data_lake}/bronze/ecommerce_{entity_name}"
path_target = f"abfs://{container_name_data_lake}/silver/ecommerce_{entity_name}"
path_quarantine = f"abfs://{container_name_data_lake}/quarantine/ecommerce_{entity_name}"
control_file_path = f"control/silver/ecommerce_{entity_name}/control_file.json"
check_interval = 10 
snapshot_id = 0

storage_options = {
    "storage_account_name": storage_account_name,
    "tenant_id": tenant_id,
    "client_id": client_id,
    "client_secret": client_secret
}

In [0]:
while True:

    snapshot_id += 1

    log.info(f"Iniciando ciclo {snapshot_id} para entidade {entity_name}.")

    log.info("Lendo Bronze") 

    df_bronze = read_delta_to_spark_df(
        path=path_source,
        storage_options= storage_options
    )

    log.info("Bronze lida com sucesso")

    log.info("Identificando arquivos da Bronze")

    all_files = [
        row.bronze_source_file
        for row in (
            df_bronze
            .select("bronze_source_file")
            .distinct()
            .collect()
        )
    ]

    processed_files = load_processed_files(
        container_client=container_client_data_lake,
        control_file_path=control_file_path
    )

    files_to_process = [
        file_path
        for file_path in all_files
        if file_path not in processed_files
    ]

    num_files = len(files_to_process)

    # Encerra o ciclo caso não tenha arquivos novos para processar

    if num_files == 0:
        log.info("Não há novos arquivos para processar.")

        time.sleep(check_interval)
        continue

    try:

        # Captura os dados a serem processados

        df_snapshot = (
            df_bronze
            .filter(
                col("bronze_source_file").isin(files_to_process)
            )
        )

        records_before = df_snapshot.count()

        log.info(f"Iniciando processamento de {records_before} registros.")

        # Aplica as regras de validação

        df_snapshot, df_quarantine = apply_silver_validation_rules(
            entity_name = entity_name,
            df_snapshot = df_snapshot,
            path_target=path_target,
            storage_options=storage_options
        )
        
        # Adiciona colunas de auditoria

        df_snapshot = add_silver_audit_columns(
            df_snapshot
        )

        records_after = df_snapshot.count()

        if records_after > 0:
            
            # Grava os dados processados

            write_adls_partitioned(
                df=df_snapshot,
                path=path_target,
                storage_options=storage_options,
                partition_by=[
                    "processed_year",
                    "processed_month",
                    "processed_day",
                    "processed_hour"
                ],
                mode="append"
            )

            log.info("Gravação na Silver concluída.")

        else:
            log.warning("Nenhum registro válido encontrado no ciclo.")

        if df_quarantine is not None:

            # Adiciona dados inválidos na quarentena

            write_adls(
                df=df_quarantine,
                path=path_quarantine,
                storage_options=storage_options,
                mode="append"
            )

            log.info("Gravação na Quarantine concluída.")

        # Atualiza o arquivo de controle

        for file_path in files_to_process:
            save_processed_file(
                container_client=container_client_data_lake,
                control_file_path=control_file_path,
                file_path=file_path
            )

        log.info(f"Ciclo {snapshot_id} processado com sucesso. {records_after}/{records_before} registros válidos.")

    except Exception as e:
        
         # Captura o erro em caso de falha no processamento do arquivo

        log.error(f"Erro no ciclo {snapshot_id}: {str(e)}")

    # Aguarda a próxima janela de processamento

    time.sleep(check_interval)

TESTE

In [0]:
# Ler silver

df_silver = read_delta_to_spark_df(
    path=path_target,
    storage_options=storage_options
)

display(df_silver)

In [0]:
# Ler quarentena

df_quarantine = read_delta_to_spark_df(
    path=path_quarantine,
    storage_options=storage_options
)

display(df_quarantine)

In [0]:
# Ler arquivo de controle
file_client = container_client_data_lake.get_file_client(control_file_path)

content = file_client.download_file().readall()

print(content.decode("utf-8"))

In [0]:
# lista arquivos da fonte
all_files = list_files(
                    container_client = container_client_data_lake,
                    folder_name = folder_name_source
)                    
print(f"print files in container:")
for files in all_files:
    print(files)

In [0]:
# lista arquivos do destino
folder_name_target = f"silver/ecommerce_{entity_name}"

all_files = list_files(
                    container_client = container_client_data_lake,
                    folder_name = folder_name_target
)                    
print(f"print files in container:")
for files in all_files:
    print(files)

In [0]:
# Limpar arquivo de controle
file_client = container_client_data_lake.get_file_client(control_file_path)

file_client.upload_data(
    "[]",
    overwrite=True
)

print("Arquivo de controle limpo.")

In [0]:
# Deletar dados da silver
container_client_data_lake.delete_directory(f"silver/ecommerce_{entity_name}")

In [0]:
# Deletar dados da quarentena
container_client_data_lake.delete_directory("silver/quarantine/ecommerce_enderecos")

In [0]:
# lista arquivos do destino


all_files = list_files(
                    container_client = container_client_data_lake,
                    folder_name= "silver"
)                    
print(f"print files in container:")
for files in all_files:
    print(files)